In [1]:
import sys, os, json, pickle
import pandas as pd
from pathlib import Path
import torch
from sklearn.metrics import r2_score

sys.path.append('../../src/fluprofiler')

from experiment_tools import (
    find_repo_root, default_exp_id, make_run_dirs, generate_matrix,
    load_data_and_dataloaders, evaluate_step
)

## inference

In [18]:
model_root = "/home/chenyh/workspace/fluProfiler/runs/reverse_tests/2023NH/v3_2/20260311_035456__v3_2_cached__pid2718573"

# season = model_root.split('/')[-3]
season = "2023SH"
latest_file = max(Path(model_root + '/checkpoints').glob('*.pth'))
model_path = str(latest_file)

In [19]:
# ---------- 1) 路径/数据 ----------
_CWD = Path.cwd().resolve()
_REPO_ROOT = find_repo_root(_CWD)     # notebook 下用 cwd 找 repo root
root_path = str(_REPO_ROOT) + "/"

data_path = root_path + "data/reverse_test/"
season_path = f"processed/test_{season}/"

exp_id = os.environ.get("FLUPROFILER_EXP_ID") or default_exp_id(_CWD, _REPO_ROOT)
tag = os.environ.get("FLUPROFILER_TAG") or "v0_1"
run_paths = make_run_dirs(_REPO_ROOT, exp_id=exp_id, tag=tag)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

data_loaders = load_data_and_dataloaders(
    data_path=data_path,
    season_path=season_path,
    batch_size=8,
    sample_limit=None,
    use_artificial=False,
    test_only=True
)
test_dataloader = data_loaders["test_dataloader"]
emb_dict = data_loaders["emb_dict"]

Loading tensor: 100%|██████████| 832/832 [00:30<00:00, 27.15file/s]


In [20]:
model = torch.load(model_path, map_location=device, weights_only=False)

model.eval()
with torch.no_grad():
    test_metrics = evaluate_step(model, test_dataloader, emb_dict, device, generate_matrix, return_predictions=True)

MAE: 1.00457
MSE: 1.58447
pearson correlation: 0.70496
spearman correlation: 0.61271
R2_score: 0.45630


## Analysis

In [19]:
test_data = pd.read_csv(f'/home/chenyh/workspace/fluProfiler/data/reverse_test/processed/test_{season}/test.csv')
test_data['prediction'] = test_metrics['predictions']
test_data['season'] = season

print('H1N1 samples: ', len(test_data[test_data['Type'] == 'H1N1']))
print('H3N2 samples: ', len(test_data[test_data['Type'] == 'H3N2']))

H1N1 samples:  2131
H3N2 samples:  5563


In [20]:
test_data.to_csv(f'{model_root}/predictionn_data.csv', index=False)

In [21]:
test_metrics[['mae', 'mse', 'pearson', 'spearman', 'r2']]

TypeError: unhashable type: 'list'

In [15]:
{k: v for k, v in test_metrics.items() if k in ['mae', 'mse', 'pearson', 'spearman', 'r2']}

{'mae': 0.9296656623461429,
 'mse': 1.4899713296166037,
 'pearson': 0.7571937759942068,
 'spearman': 0.6954969213685277,
 'r2': 0.49828138583201287}